## 1. Important Imports

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd 
import numpy as np 
import seaborn as se 
import matplotlib as plt
import random 

## 2. Createting PySpark Session

In [0]:
spark = SparkSession.builder.appName('Data_Analysis_with_PySpark').getOrCreate()

## 3. Generating Data

In [0]:
names = [
    "Alice", "Bob", "Charlie", "David", "Eve", "Fiona", "George", "Hannah",
    "Ivy", "Jack", "Kaitlyn", "Liam", "Olivia", "Liam", "Emma", "Noah", 
    "Ava", "Oliver", "Charlotte", "Elijah", "Sophia", "James", "Amelia", 
    "Benjamin", "Isabella", "Lucas", "Mia", "Mason", "Harper", "Ethan", 
    "Evelyn", "Alexander", "Abigail", "Henry", "Ella", "Jackson", "Scarlett", 
    "Aiden", "Grace", "Samuel", "Lily", "Sebastian"
]
genders = ["Male", "Female", None]
subjects = ["Math", "Science", "History", "English", "Art", "PE", None]
cities = [
    "New York", "Los Angeles", "Chicago", "Houston", 
    "Bangalore", "Hajipur", "Sitamardhi", "MP", None
]
states = ["NY", "CA", "IL", "TX", "Bihar", "Karnataka", "Sitamardhi", None]
countries = ["USA", "India", "Pakistan", "Nepal", "China", None]
graduated_status = ["Yes", "No", None]

data = [
    (
        i, 
        random.choice(names),  # student_name
        random.choice([random.randint(18, 25), None]),  # age
        random.choice(genders),  # gender
        random.choice(subjects),  # subject
        random.choice([random.randint(50, 100), None]),  # marks
        random.choice(cities),  # city
        random.choice(states),  # state
        random.choice(countries),  # country
        random.choice(graduated_status),  # graduated
    )
    for i in range(1, 501)
]

## 4. Creating a DATAFRAME

In [0]:
schema = StructType([
    StructField("student_id", IntegerType(), True),
    StructField("student_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("marks", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("graduated", StringType(), True),
])

df = spark.createDataFrame(data,schema=schema)
df.show(5)

+----------+------------+----+------+-------+-----+-----------+---------+-------+---------+
|student_id|student_name| age|gender|subject|marks|       city|    state|country|graduated|
+----------+------------+----+------+-------+-----+-----------+---------+-------+---------+
|         1|        Lily|  24|Female|Science|   58|         MP|    Bihar|  China|       No|
|         2|    Scarlett|null|  Male|English| null|Los Angeles|Karnataka|  China|     null|
|         3|       James|  25|Female|   null|   73|       null|       NY|  Nepal|      Yes|
|         4|         Ivy|null|  Male|English| null|    Chicago|       NY|    USA|      Yes|
|         5|      Oliver|  19|Female|History| null|  Bangalore|       TX|  India|      Yes|
+----------+------------+----+------+-------+-----+-----------+---------+-------+---------+
only showing top 5 rows



## 5.OverView of DataFrame

In [0]:
df.show(10)

+----------+------------+----+------+-------+-----+-----------+---------+--------+---------+
|student_id|student_name| age|gender|subject|marks|       city|    state| country|graduated|
+----------+------------+----+------+-------+-----+-----------+---------+--------+---------+
|         1|        Lily|  24|Female|Science|   58|         MP|    Bihar|   China|       No|
|         2|    Scarlett|null|  Male|English| null|Los Angeles|Karnataka|   China|     null|
|         3|       James|  25|Female|   null|   73|       null|       NY|   Nepal|      Yes|
|         4|         Ivy|null|  Male|English| null|    Chicago|       NY|     USA|      Yes|
|         5|      Oliver|  19|Female|History| null|  Bangalore|       TX|   India|      Yes|
|         6|      Evelyn|  21|  null|History| null|  Bangalore|    Bihar|Pakistan|     null|
|         7|      Samuel|  22|Female|History| null|  Bangalore|       CA|   Nepal|     null|
|         8|     Charlie|null|  null|     PE| null|    Houston|       

In [0]:
df.describe().show()

+-------+-----------------+------------+-----------------+------+-------+------------------+----------+-----+-------+---------+
|summary|       student_id|student_name|              age|gender|subject|             marks|      city|state|country|graduated|
+-------+-----------------+------------+-----------------+------+-------+------------------+----------+-----+-------+---------+
|  count|              500|         500|              251|   356|    430|               249|       447|  446|    414|      350|
|   mean|            250.5|        null|21.53784860557769|  null|   null| 75.63052208835342|      null| null|   null|     null|
| stddev|144.4818327679989|        null|2.371826670097975|  null|   null|14.479408074713088|      null| null|   null|     null|
|    min|                1|     Abigail|               18|Female|    Art|                50| Bangalore|Bihar|  China|       No|
|    max|              500|      Sophia|               25|  Male|Science|               100|Sitamardhi| 

In [0]:
df.dtypes

Out[7]: [('student_id', 'int'),
 ('student_name', 'string'),
 ('age', 'int'),
 ('gender', 'string'),
 ('subject', 'string'),
 ('marks', 'int'),
 ('city', 'string'),
 ('state', 'string'),
 ('country', 'string'),
 ('graduated', 'string')]

In [0]:
df.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- student_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- subject: string (nullable = true)
 |-- marks: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- graduated: string (nullable = true)



In [0]:
## Checking Missing Values
str_column = ['student_name','gender','subject','city','state','country','graduated']
num_column = ['student_id','age','marks']

missing_val_dict = {}
for index,column in enumerate(df.columns):
    if column in str_column:
        missing_str_count = df.filter(col(column).eqNullSafe(None)\
            | col(column).isNull()
            ).count()
        missing_val_dict.update({column:missing_str_count})
    if column in num_column:
        missing_num_count = df.where(col(column).isin([0,None,np.nan])).count()
        missing_val_dict.update({column:missing_num_count})
missing_df = pd.DataFrame.from_dict([missing_val_dict])
missing_df




,student_id,student_name,age,gender,subject,marks,city,state,country,graduated
0,0,0,0,144,70,0,53,54,86,150


## 6. Replacing Null Values

In [0]:
df = df.fillna({
    'gender' : 'UniSex',
    'subject' : 'Hindi',
    'city' : 'Kalitand',
    'State' : 'Others',
    'Country' : 'India',
    'graduated' : 'Failed'
})

In [0]:
## Checking Missing Values
str_column = ['student_name','gender','subject','city','state','country','graduated']
num_column = ['student_id','age','marks']

missing_val_dict = {}
for index,column in enumerate(df.columns):
    if column in str_column:
        missing_str_count = df.filter(col(column).eqNullSafe(None)\
            | col(column).isNull()
            ).count()
        missing_val_dict.update({column:missing_str_count})
    if column in num_column:
        missing_num_count = df.where(col(column).isin([0,None,np.nan])).count()
        missing_val_dict.update({column:missing_num_count})
missing_df = pd.DataFrame.from_dict([missing_val_dict])
missing_df




,student_id,student_name,age,gender,subject,marks,city,state,country,graduated
0,0,0,0,0,0,0,0,0,0,0


## 7. Checking duplicate Values in each column

In [0]:
for index,column in enumerate(df.columns):
    print(f"checking duplicates in the column: {column}")
    duplicate_count = df.groupBy(column).count().filter("count > 1 ")
    duplicate_count.show()


checking duplicates in the column: student_id
+----------+-----+
|student_id|count|
+----------+-----+
+----------+-----+

checking duplicates in the column: student_name
+------------+-----+
|student_name|count|
+------------+-----+
|       Lucas|    8|
|       Grace|   10|
|         Ivy|   11|
|       James|   18|
|    Benjamin|   16|
|      Hannah|   12|
|        Jack|    9|
|         Ava|   18|
|        Ella|   14|
|      Evelyn|   16|
|        Noah|   13|
|       Ethan|   20|
|     Charlie|   14|
|         Mia|    7|
|        Liam|   21|
|      Elijah|   16|
|      Samuel|   10|
|   Alexander|   14|
|       Alice|   12|
|     Kaitlyn|   13|
+------------+-----+
only showing top 20 rows

checking duplicates in the column: age
+----+-----+
| age|count|
+----+-----+
|  22|   16|
|null|  249|
|  19|   29|
|  23|   30|
|  25|   36|
|  24|   36|
|  21|   33|
|  18|   32|
|  20|   39|
+----+-----+

checking duplicates in the column: gender
+------+-----+
|gender|count|
+------+-----+
|Fe

In [0]:
duplicate_counts = []
# Loop through each column in the DataFrame
for column in df.columns:
    # Group by the column and count duplicates (count > 1)
    dup_count = df.groupBy(column).count().filter("count > 1")
    # Get the number of duplicate values
    count = dup_count.count()
    # Append the result as a tuple (column name, duplicate count)
    duplicate_counts.append((column, count))
# Convert the list to a Pandas DataFrame
dup_df = pd.DataFrame(duplicate_counts, columns=['Column_Name', "Dup_count"])
# Show the Pandas DataFrame
dup_df


,Column_Name,Dup_count
0,student_id,0
1,student_name,41
2,age,9
3,gender,3
4,subject,7
5,marks,48
6,city,9
7,state,8
8,country,5
9,graduated,3


In [0]:
# Loop through each column in the DataFrame
for column in df.columns:
    print(column)
    dup_count = df.groupBy(column).count().filter("count > 1")
    # Get the number of duplicate values
    count = dup_count.count()
    # Append the result as a tuple (column name, duplicate count)
    duplicate_counts.append((column, count))
# Convert the list to a Pandas DataFrame
dup_df = pd.DataFrame(duplicate_counts, columns=['Column_Name', "Dup_count"])
# Show the Pandas DataFrame
dup_df


student_id
student_name
age
gender
subject
marks
city
state
country
graduated


,Column_Name,Dup_count
0,student_id,0
1,student_name,41
2,age,9
3,gender,3
4,subject,7
5,marks,48
6,city,9
7,state,8
8,country,5
9,graduated,3


## 8.Descriptive Statistics and Basic Summarization

In [0]:
df.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- student_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = false)
 |-- subject: string (nullable = false)
 |-- marks: integer (nullable = true)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- country: string (nullable = false)
 |-- graduated: string (nullable = false)



In [0]:
# What are the central tendencies (mean, median, mode) of marks and age?
df.groupBy("subject").agg(
    mean("marks").alias("marks_mean"),
    mean("age").alias("mean_age")
).show()



+-------+-----------------+------------------+
|subject|       marks_mean|          mean_age|
+-------+-----------------+------------------+
|Science|72.63333333333334| 21.63888888888889|
|    Art|             78.0| 21.58823529411765|
|   Math| 75.3695652173913|21.333333333333332|
|English|           76.225|            21.375|
|History|75.21052631578948|21.516129032258064|
|  Hindi|76.22222222222223|21.571428571428573|
|     PE|75.83870967741936|21.818181818181817|
+-------+-----------------+------------------+



In [0]:
# What is the spread of marks in each subject?
df.groupBy("subject").agg(
    min("marks").alias("min_marks"),
    max("marks").alias("max_marks"),
).show()

+-------+---------+---------+
|subject|min_marks|max_marks|
+-------+---------+---------+
|Science|       50|      100|
|    Art|       50|       98|
|   Math|       50|      100|
|English|       53|       99|
|History|       54|      100|
|  Hindi|       50|      100|
|     PE|       50|       99|
+-------+---------+---------+



In [0]:
# What is the standard deviation of marks for different cities or countries?
df.groupBy("city").agg(
    stddev("marks").alias("stddev_marks")
).show()

+-----------+------------------+
|       city|      stddev_marks|
+-----------+------------------+
|  Bangalore|15.403745074456424|
|Los Angeles|15.480198937345248|
| Sitamardhi|15.532516150676118|
|    Chicago|  13.9485759331471|
|    Hajipur|14.422342675653224|
|    Houston|12.725126865966434|
|   New York|14.101084345189378|
|         MP|15.028468918981083|
|   Kalitand|13.589458675634367|
+-----------+------------------+



In [0]:
from pyspark.sql import functions as F

# Calculate the distribution of students in each subject
df.groupBy("subject").agg(
    F.count("student_id").alias("stu_count")  # Count the number of students in each subject
).show()


+-------+---------+
|subject|stu_count|
+-------+---------+
|Science|       70|
|    Art|       68|
|   Math|       83|
|English|       74|
|History|       69|
|  Hindi|       70|
|     PE|       66|
+-------+---------+



In [0]:
# How many students belong to each gender and how does their academic performance differ?
df.groupBy("gender").agg(
    F.count("*").alias("stu_count"),       # Count of students per gender
    F.avg("marks").alias("avg_marks"),     # Average marks per gender
    F.stddev("marks").alias("std_marks")   # Standard deviation of marks per gender
).show()


+------+---------+-----------------+------------------+
|gender|stu_count|        avg_marks|         std_marks|
+------+---------+-----------------+------------------+
|Female|      154|75.11392405063292|15.392295226935202|
|UniSex|      144|76.20289855072464|13.954315914845894|
|  Male|      202|75.64356435643565|14.222928079981171|
+------+---------+-----------------+------------------+



In [0]:
# What is the distribution of students across different cities, states, and countries?
df.groupBy("city","state","country").count().show()

+-----------+----------+--------+-----+
|       city|     state| country|count|
+-----------+----------+--------+-----+
|Los Angeles| Karnataka|   China|    2|
| Sitamardhi|     Bihar|   China|    3|
|         MP|        NY|   India|    5|
|   New York| Karnataka|   India|    2|
|   New York|        TX|   China|    1|
|    Chicago|        CA|   China|    3|
|    Chicago|        NY|   China|    4|
|   Kalitand|        NY|   India|    3|
|   New York|Sitamardhi|     USA|    2|
|         MP|Sitamardhi|Pakistan|    2|
|         MP|        IL|   India|    3|
| Sitamardhi|        TX|Pakistan|    3|
|Los Angeles|        IL|     USA|    1|
|  Bangalore|        IL|     USA|    5|
|         MP|     Bihar|Pakistan|    2|
|         MP|     Bihar|   China|    1|
| Sitamardhi|        CA|   India|    3|
|         MP| Karnataka|Pakistan|    1|
|         MP|        NY|     USA|    3|
|  Bangalore|        TX|   India|    2|
+-----------+----------+--------+-----+
only showing top 20 rows



In [0]:
# How many students are marked as graduated versus those who are not? What is the graduation rate?
graduation_stats = df.groupBy("graduated").count()
total_student = df.count()
graduation_stats.withColumn("graduation_rate",col(total_student)/graduation_stats *100).show()


---------------------------------------------------------------------------
Py4JError                                 Traceback (most recent call last)
File <command-4380487821440187>:4
      2 graduation_stats = df.groupBy("graduated").count()
      3 total_student = df.count()
----> 4 graduation_stats.withColumn("graduation_rate",col(total_student)/graduation_stats *100).show()

File /databricks/spark/python/pyspark/sql/utils.py:164, in try_remote_functions.<locals>.wrapped(*args, **kwargs)
    162     return getattr(functions, f.__name__)(*args, **kwargs)
    163 else:
--> 164     return f(*args, **kwargs)

File /databricks/spark/python/pyspark/sql/functions.py:192, in col(col)
    165 @try_remote_functions
    166 def col(col: str) -> Column:
    167     """
    168     Returns a :class:`~pyspark.sql.Column` based on the given column name.
    169 
   (...)
    190     Column<'x'>
    191     """
--> 192     return _invoke_function("col", col)

File /databricks/spark/python/pyspark